# **🇹🇭 Constituency OCR: Official PaddleOCR-VL Pipeline**

This notebook uses the official `paddleocr` library to run the **PaddleOCR-VL** Vision Language Model. This avoids the Hugging Face `transformers` compatibility issues and utilizes the state-of-the-art document parsing pipeline directly from PaddlePaddle.

### 🛠️ Setup Instructions:
We must install `paddlepaddle-gpu` and the latest `paddleocr` packages. Make sure Kaggle's Internet is **ON** and the Accelerator is set to **GPU (T4 x 2 or P100)**.

In [ ]:
!python -m pip install paddlepaddle-gpu>=2.5.0.post118 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
!pip install -q "paddleocr>=2.9.0" rapidfuzz textdistance pandas pillow

In [ ]:
import os
import re
import json
import glob
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from rapidfuzz import process, fuzz
import textdistance
import gc

# Import the new VLM pipeline from PaddleOCR
from paddleocr import PaddleOCRVL

## 1. Loader functions & Initialization

In [ ]:
DATA_DIR = './data'     
IMG_DIR = os.path.join(DATA_DIR, 'images')
TEMPLATE_PATH = os.path.join(DATA_DIR, 'submission_template.csv')
PADDLE_OUT_DIR = './paddle_temp_out'

os.makedirs(PADDLE_OUT_DIR, exist_ok=True)

sub_df = pd.read_csv(TEMPLATE_PATH)
doc_ids = sub_df['doc_id'].unique()
print(f"Loaded {len(sub_df)} rows for {len(doc_ids)} unique documents.")

def get_image_paths_for_doc(doc_id):
    """Reads all PNG page paths for a given doc_id."""
    images = []
    base_img = os.path.join(IMG_DIR, f"{doc_id}.png")
    if os.path.exists(base_img):
        images.append(base_img)
    
    page = 2
    while True:
        page_img = os.path.join(IMG_DIR, f"{doc_id}_page{page}.png")
        if os.path.exists(page_img):
            images.append(page_img)
            page += 1
        else:
            break
    return images

def clean_vote_string(s):
    """Removes non-digits and converts Thai numerals to Arabic."""
    thai_to_arabic = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
    s = str(s).translate(thai_to_arabic)
    s = re.sub(r'\D', '', s)
    return s if s else "0"

print("Initializing PaddleOCR-VL model into GPU...")
pipeline = PaddleOCRVL(pipeline_version="v1")
print("Loaded Successfully!")

## 2. Table Extraction & Fuzzy Extraction Logic
Since `PaddleOCRVL` extracts full document structure (usually saving Markdown or JSON), we run it on the election papers, read the Markdown output, and intelligently parse the table.

In [ ]:
def extract_votes_paddle_md(doc_id, parties_list):
    images = get_image_paths_for_doc(doc_id)
    if not images:
        return {p: "0" for p in parties_list}
    
    final_extracted_map = {p: "0" for p in parties_list}
    
    for page_idx, img_path in enumerate(images):
        try:
            output = pipeline.predict(img_path)
            
            for res in output:
                # We save it as markdown to standard folder, so we know where to look
                md_save_path = os.path.join(PADDLE_OUT_DIR, f"{doc_id}_page_{page_idx}.md")
                # Res object has save_to_markdown or text attribute.
                # The official docs use: res.save_to_markdown(save_path="/tmp") 
                # Note: save_to_markdown automatically appends things. Let's try getting the text if possible.
                full_markdown = getattr(res, "text", "") 
                if not full_markdown:
                     full_markdown = str(res) # Fallback if text isn't a direct attribute

                # Parse the raw markdown content line by line
                lines = full_markdown.split('\n')
                
                for party in parties_list:
                    if final_extracted_map[party] != "0":
                        continue # Found earlier
                        
                    best_score = 0
                    best_line = ""
                    
                    # Scanning the Markdown table rows to find the best party match
                    for line in lines:
                        score = fuzz.partial_ratio(party, line)
                        if score > best_score:
                            best_score = score
                            best_line = line
                            
                    if best_score > 80:
                        # Excellent! Now let's extract digits from that markdown row
                        # Typically a table row looks like: | 5 | พรรคเพื่อไทย | 15530 |
                        columns = [c.strip() for c in best_line.split('|') if c.strip()]
                        
                        candidate_numbers = []
                        for col in columns:
                            clean_digits = clean_vote_string(col)
                            if clean_digits != "0" and len(clean_digits) > 0:
                                candidate_numbers.append(clean_digits)
                                
                        if candidate_numbers:
                            # In Thai election forms, the vote count is consistently the right-most digit column
                            final_extracted_map[party] = candidate_numbers[-1]
                            
        except Exception as e:
            print(f"PaddleOCRVL Error on {doc_id} Page {page_idx}: {e}")
            
    return final_extracted_map


## 3. Evaluation on Samples

In [ ]:
def evaluate_on_samples():
    label_files = glob.glob(os.path.join(DATA_DIR, "sample_labels", "*.json"))
    total_dist = 0
    total_rows = 0
    
    print(f"Evaluating on {len(label_files)} models using PaddleOCRVL Markdown Table Parser...")
    for lpath in label_files:
        with open(lpath, 'r', encoding='utf-8') as f:
            gt_data = json.load(f)
            
        doc_id = os.path.basename(lpath).replace('.json', '')
        gt_map = {item['party']: str(item['votes']) for item in gt_data['results']}
        
        pred_map = extract_votes_paddle_md(doc_id, list(gt_map.keys()))
        
        for party, gt_votes in gt_map.items():
            pred_votes = pred_map.get(party, "0")
            dist = textdistance.levenshtein(gt_votes, pred_votes)
            total_dist += dist
            total_rows += 1
            
    if total_rows > 0:
        print(f"Mean Levenshtein Distance: {total_dist / total_rows:.4f}")
        
# evaluate_on_samples()

## 4. Generative Final Submission CSV

In [ ]:
results_list = []

# Running on 3 docs for testing, remove [:3] for complete processing
for doc_id in tqdm(doc_ids[:3], desc="Processing Documents (PaddleOCRVL)"):
    parties_to_find = sub_df[sub_df['doc_id'] == doc_id]['party_name'].tolist()
    
    extracted_map = extract_votes_paddle_md(doc_id, parties_to_find)
    
    for party in parties_to_find:
        results_list.append({
            'doc_id': doc_id,
            'party_name': party,
            'votes': extracted_map.get(party, "0")
        })
        
for res in results_list:
    mask = (sub_df['doc_id'] == res['doc_id']) & (sub_df['party_name'] == res['party_name'])
    sub_df.loc[mask, 'votes'] = res['votes']

sub_df.to_csv('submission_paddleocrvl.csv', index=False)
print("Done! Saved CSV.")
sub_df.head(10)